This notebook first describes uninformed
sampling methods including random, grid, and quasi-random (e.g., Latin hypercube, Sobol
sequences) and shows how quasi-random achieves a much more even distribution of points.
Finally, we compare the efficiency of these traditional methods for design of
experiments (DoE) against Bayesian optimization. We show that, on average, Bayesian
optimization is much more efficient than traditional DoE.

In [40]:
import numpy as np
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
from scipy.stats import qmc
from os import path

In [41]:
bounds = {"metallicity": [0.0001,0.03], "common_alpha": [0.,10.], "sigma_bh": [0.,1000.], "sigma_sn": [0.,1000.]}
num_samples = 1000
num_digits = 5

## Uninformed Sampling Methods

These are sampling methods that do not incorporate information about the objective function
to be optimized. We will cover grid search, random search, and two quasi-random methods:
Latin hypercube and Sobol sequences.

### Grid

Grid sampling is a structured approach to sampling from a search space. It involves
creating a grid of points over the space, and then selecting points from the grid. When
grids are used for certain types of problems (e.g., finite element methods), the
algorithms are often more straightforward than for non-lattice point sets. This can be
an effective way to ensure that the entire search space is explored, but it can also
lead to a lot of unnecessary points being generated, especially for high-dimensional
search spaces, due to large, systematic "pockets" in the search space.


In [42]:
from sklearn.model_selection import ParameterGrid

def get_grid_samples(bounds, num_samples = 10, seed=None):
    # seed is unused, for compatibility only
    param_grid = {}
    num_pts_per_dim = max(1, np.floor(num_samples ** (1 / len(bounds))).astype(int))
    for name, bnd in bounds.items():
        param_grid[name] = np.linspace(bnd[0], bnd[1], num=num_pts_per_dim)
    return pd.DataFrame(list(ParameterGrid(param_grid)))

def get_grid_samples_extended(bounds, num_samples=[10,10], seed=None):
    grid_samples_1 = get_grid_samples(bounds, num_samples[0],seed=seed)
    grid_samples_2 = get_grid_samples(bounds, num_samples[1],seed=seed)
    return pd.concat([grid_samples_1,grid_samples_2],ignore_index=True,sort=False)

grid_samples = get_grid_samples(bounds, num_samples,seed=None)
grid_samples

,common_alpha,metallicity,sigma_bh,sigma_sn
0,0.0,0.0001,0.0,0.0
1,0.0,0.0001,0.0,250.0
2,0.0,0.0001,0.0,500.0
3,0.0,0.0001,0.0,750.0
4,0.0,0.0001,0.0,1000.0
...,...,...,...,...
620,10.0,0.0300,1000.0,0.0
621,10.0,0.0300,1000.0,250.0
622,10.0,0.0300,1000.0,500.0
623,10.0,0.0300,1000.0,750.0


In [43]:
grid_fig = px.scatter(grid_samples, x="metallicity", y="common_alpha", width=400, height=400)
grid_fig


### Random

Random sampling is a simple and straightforward method for generating samples from a
search space. It involves randomly selecting points from the space, without any regard
for their distribution. This can be an effective method for exploring search spaces, and
it is often more effective than grid search. While it doesn't have the large, systematic
pockets characteristic of grid search, it has large, occasional gaps due to the random
nature of the search.

In [44]:
from numpy.random import default_rng

def get_random_samples(bounds, num_samples=9, seed=None):
    rng = default_rng(seed)
    samples = {}
    for parameter, bound in bounds.items():
        samples[parameter] = rng.uniform(bound[0], bound[1], num_samples)
        samples[parameter] = np.round(samples[parameter],num_digits)
    return pd.DataFrame(samples)

random_samples = get_random_samples(bounds, num_samples,seed=None)
random_samples


,metallicity,common_alpha,sigma_bh,sigma_sn
0,0.01318,7.20603,49.00611,560.71082
1,0.02327,3.03391,792.77917,610.39564
2,0.00843,1.32855,474.35610,762.15095
3,0.00127,7.54076,591.27848,734.72903
4,0.00829,5.90141,304.29450,453.91511
...,...,...,...,...
995,0.01173,4.55813,170.87457,323.72668
996,0.00341,8.79714,995.52334,870.95486
997,0.02923,2.88108,960.89136,577.51653
998,0.02307,3.98748,323.28094,401.79935


In [45]:
random_fig = px.scatter(random_samples,x="metallicity", y="common_alpha", width=400, height=400)
random_fig

## Quasi-Random

Quasi-random sampling methods, also known as low-discrepancy or deterministic sampling methods, are a family of sampling techniques that are designed to
produce samples that are more evenly distributed than random samples. Unlike random
sampling, which selects points randomly and independently, quasi-random sampling methods
aim to achieve a more uniform coverage of the parameter space by reducing the
**discrepancy** between the generated points and the desired distribution.

We'll discuss two common quasi-random sampling methods: Latin hypercube and
Sobol sequences.

### Latin Hypercube

Latin hypercube sampling (LHS) is a variation of grid sampling that aims to improve the
uniformity of the samples. It does this by ensuring that each dimension of the search
space is represented equally in the sample set. It involves dividing the parameter space
into equally spaced intervals and randomly selecting one point within each interval. LHS
ensures a more even coverage of the parameter space compared to random or grid sampling
methods (i.e., lower discrepancy).

In [46]:
def get_latin_hypercube_samples(bounds, num_samples=10, seed=None):
    sampler = qmc.LatinHypercube(d=len(bounds), optimization="random-cd", seed=seed)
    samples = sampler.random(num_samples)
    l_bounds = [bound[0] for bound in bounds.values()]
    u_bounds = [bound[1] for bound in bounds.values()]
    samples = qmc.scale(samples, l_bounds, u_bounds)
    samples=np.round(samples,num_digits)
    return pd.DataFrame(samples, columns=list(bounds.keys()))

latin_hypercube_samples = get_latin_hypercube_samples(bounds, num_samples,seed=None)
latin_hypercube_samples


,metallicity,common_alpha,sigma_bh,sigma_sn
0,0.00616,2.16364,873.96402,850.16329
1,0.00382,0.22853,481.58268,990.10194
2,0.01993,3.96646,950.94287,431.25297
3,0.01827,6.48929,59.37444,143.43668
4,0.01061,4.51283,249.34521,625.86066
...,...,...,...,...
995,0.01213,7.69264,541.00734,722.61788
996,0.01641,4.29892,263.01122,448.56027
997,0.01207,9.18018,492.84989,259.06066
998,0.00565,2.28625,440.20007,935.72349


In [47]:
latin_hypercube_fig = px.scatter(
    latin_hypercube_samples, x="metallicity", y="common_alpha", width=400, height=400
)
latin_hypercube_fig


In [48]:
with open("initial_samples_latin_hypercube.sh", "w") as f:
    for idx, row in latin_hypercube_samples.iterrows():
        row_id = f"{idx:04d}"  # ZERO-PAD TO 3 DIGITS

        command = (
            f"python cli.py "
            f"--metallicity={row['metallicity']} "
            f"--envelope_eff={row['common_alpha']} "
            f"--sigma_bh={int(row['sigma_bh'])} "
            f"--sigma_ns={int(row['sigma_sn'])} "
            f"--num_systems=1000"
        )
        f.write(command + "\n")

In [49]:
latin_hypercube_samples.to_csv("latin_hypercube_samples.csv", index=False)

### Sobol Sequences

Sobol sequences are another type of quasi-random sampling with good space-filling (i.e.,
low discrepancy) properties. Sobol sequences "use a base of two to form successively
finer uniform partitions of the unit interval and then reorder the coordinates in each
dimension" ([source](https://en.wikipedia.org/wiki/Sobol_sequence)). In other words, to
obtain optimal space-filling properties, sample sizes that are powers of 2 (i.e.,
$n=2^m$) should be used. For each dimension,
Sobol sequences utilize a set of direction numbers, also known as primitive polynomials,
that are converted to a binary representation and undergo bitwise operations to arrive
at the final sequence.

In [50]:
from scipy.stats.qmc import Sobol

def get_sobol_samples(bounds, num_samples=10, seed=None):
    sampler = Sobol(len(bounds), seed=seed)
    samples = sampler.random(num_samples)
    
    l_bounds = [bound[0] for bound in bounds.values()]
    u_bounds = [bound[1] for bound in bounds.values()]
    samples = qmc.scale(samples, l_bounds, u_bounds)
    samples = np.round(samples,num_digits)
    
    return pd.DataFrame(samples, columns=list(bounds.keys()))

sobol_samples = get_sobol_samples(bounds, num_samples,seed=None)
sobol_samples

/global/cfs/projectdirs/katrin/users/aschuetz/software/conda_envs/resum/lib/python3.10/site-packages/scipy/stats/_qmc.py:958: UserWarning:

The balance properties of Sobol' points require n to be a power of 2.



,metallicity,common_alpha,sigma_bh,sigma_sn
0,0.02823,0.95744,845.84340,639.75936
1,0.00430,9.75764,215.78609,434.15998
2,0.00764,2.70451,700.06446,956.72664
3,0.01734,6.42415,361.38320,218.96886
4,0.02097,4.92554,7.51005,55.38670
...,...,...,...,...
995,0.01817,9.56179,119.92373,718.79529
996,0.02201,1.82212,265.08925,555.19787
997,0.01232,8.12201,673.46306,260.79060
998,0.00149,4.49639,157.75670,847.81259


In [51]:


with open("initial_samples_sobol.sh", "w") as f:
    for idx, row in sobol_samples.iterrows():
        row_id = f"{idx:04d}"  # ZERO-PAD TO 3 DIGITS

        command = (
            f"python cli_1.py "
            f"--metallicity={row['metallicity']} "
            f"--envelope_eff={row['common_alpha']} "
            f"--sigma_bh={int(row['sigma_bh'])} "
            f"--sigma_ns={int(row['sigma_sn'])} "
            f"--num_systems=5000 "
            f"--prefix=run_lf_val "
            f"--simnumber={row_id}"
        )
        f.write(command + "\n")

In [52]:
sobol_samples.to_csv("sobol_samples.csv", index=False)

In [53]:
sobol_fig = px.scatter(sobol_samples, x="metallicity", y="common_alpha", width=400, height=400)
sobol_fig

### Halton Sequences

Halton sequences are 

In [54]:
def get_halton_samples(bounds, num_samples=10, seed=None):
    sampler = qmc.Halton(d=len(bounds), scramble=False, seed=seed)
    samples = sampler.random(num_samples)
    l_bounds = [bound[0] for bound in bounds.values()]
    u_bounds = [bound[1] for bound in bounds.values()]
    samples = qmc.scale(samples, l_bounds, u_bounds)
    samples=np.round(samples,num_digits)
    return pd.DataFrame(samples, columns=list(bounds.keys()))\

halton_samples = get_halton_samples(bounds, num_samples,seed=None)
halton_samples

,metallicity,common_alpha,sigma_bh,sigma_sn
0,0.00010,0.00000,0.00,0.00000
1,0.01505,3.33333,200.00,142.85714
2,0.00758,6.66667,400.00,285.71429
3,0.02252,1.11111,600.00,428.57143
4,0.00384,4.44444,800.00,571.42857
...,...,...,...,...
995,0.02343,8.53681,195.52,201.99917
996,0.00474,2.98125,395.52,344.85631
997,0.01969,6.31459,595.52,487.71345
998,0.01222,9.64792,795.52,630.57060


In [55]:
halton_fig = px.scatter(halton_samples, x="metallicity", y="common_alpha", width=400, height=400)
halton_fig

### Comparison between sampling methods

The quasi-random methods tend to have better space-filling properties than random or
grid search. Note the large systematic gaps in grid, the large occasional gaps in
random, and the more even distribution of points in LHS and Sobol.

In [56]:
sampling_fns = dict(
    grid=get_grid_samples,
    random=get_random_samples,
    latin_hypercube=get_latin_hypercube_samples,
    sobol=get_sobol_samples,
    halton=get_halton_samples,
)

sample_nums = [5,50,100,250,500]
sample_nums.reverse()
        
sample_dfs = []
for name, sampling_fn in sampling_fns.items():
    for num_samples in sample_nums:
        sample_df = sampling_fn(bounds, num_samples)
        sample_df["name"] = name
        sample_df["num_samples"] = np.sum(num_samples)
        sample_dfs.append(sample_df)

compare_df = pd.concat(sample_dfs, names=list(bounds.keys()),axis=0)

/global/cfs/projectdirs/katrin/users/aschuetz/software/conda_envs/resum/lib/python3.10/site-packages/scipy/stats/_qmc.py:958: UserWarning:

The balance properties of Sobol' points require n to be a power of 2.

/global/cfs/projectdirs/katrin/users/aschuetz/software/conda_envs/resum/lib/python3.10/site-packages/scipy/stats/_qmc.py:958: UserWarning:

The balance properties of Sobol' points require n to be a power of 2.

/global/cfs/projectdirs/katrin/users/aschuetz/software/conda_envs/resum/lib/python3.10/site-packages/scipy/stats/_qmc.py:958: UserWarning:

The balance properties of Sobol' points require n to be a power of 2.

/global/cfs/projectdirs/katrin/users/aschuetz/software/conda_envs/resum/lib/python3.10/site-packages/scipy/stats/_qmc.py:958: UserWarning:

The balance properties of Sobol' points require n to be a power of 2.

/global/cfs/projectdirs/katrin/users/aschuetz/software/conda_envs/resum/lib/python3.10/site-packages/scipy/stats/_qmc.py:958: UserWarning:

The balance prop

In [57]:
compare_df

,common_alpha,metallicity,sigma_bh,sigma_sn,name,num_samples
0,0.00000,0.00010,0.000000,0.000000,grid,500
1,0.00000,0.00010,0.000000,333.333333,grid,500
2,0.00000,0.00010,0.000000,666.666667,grid,500
3,0.00000,0.00010,0.000000,1000.000000,grid,500
4,0.00000,0.00010,333.333333,0.000000,grid,500
...,...,...,...,...,...,...
0,0.00000,0.00010,0.000000,0.000000,halton,5
1,3.33333,0.01505,200.000000,142.857140,halton,5
2,6.66667,0.00758,400.000000,285.714290,halton,5
3,1.11111,0.02252,600.000000,428.571430,halton,5


In [58]:
fig = px.scatter(
    compare_df,
    x="metallicity", 
    y="common_alpha",
    facet_row="num_samples",
    facet_col="name",
    width=1000,
    height=1000,
)
fig.show()

### Worsening performance in higher dimensions

As we observe the discrepancy associated with sampling methods in higher dimensions, we
notice that the gap between the quasi-random methods and the random and grid methods
widens. In other words, quasi-random methods increasingly outperform random and grid as
the dimensionality increases.

In [59]:

three_grid= get_grid_samples(
    dict(metallicity=bounds["metallicity"], common_alpha=bounds["common_alpha"], sigma_bh=bounds["sigma_bh"], sigma_sn=bounds["sigma_sn"]), num_samples=50
)
three_random = get_sobol_samples(
    dict(metallicity=bounds["metallicity"], common_alpha=bounds["common_alpha"], sigma_bh=bounds["sigma_bh"], sigma_sn=bounds["sigma_sn"]), num_samples=50
)
three_sobol = get_sobol_samples(
    dict(metallicity=bounds["metallicity"], common_alpha=bounds["common_alpha"], sigma_bh=bounds["sigma_bh"], sigma_sn=bounds["sigma_sn"]), num_samples=50
)
three_LH = get_latin_hypercube_samples(
    dict(metallicity=bounds["metallicity"], common_alpha=bounds["common_alpha"], sigma_bh=bounds["sigma_bh"], sigma_sn=bounds["sigma_sn"]), num_samples=50
)
three_halton = get_halton_samples(
    dict(metallicity=bounds["metallicity"], common_alpha=bounds["common_alpha"], sigma_bh=bounds["sigma_bh"], sigma_sn=bounds["sigma_sn"]), num_samples=50
)

/global/cfs/projectdirs/katrin/users/aschuetz/software/conda_envs/resum/lib/python3.10/site-packages/scipy/stats/_qmc.py:958: UserWarning:

The balance properties of Sobol' points require n to be a power of 2.



In [60]:
# https://community.plotly.com/t/plotting-a-simple-1d-number-line/39169/4
import plotly.graph_objects as go
fig = go.Figure()
x = three_grid["metallicity"]
fig.add_trace(go.Scatter(
    x=x, y=[0] * len(x), mode='markers', marker_size=20,
))
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False, 
                 zeroline=True, zerolinecolor='black', zerolinewidth=3,
                 showticklabels=False)
fig.update_layout(height=200, plot_bgcolor='white')
fig.show()

fig3 = go.Figure()
x3 = three_random["metallicity"]
fig3.add_trace(go.Scatter(
    x=x3, y=[0] * len(x3), mode='markers', marker_size=20,
))
fig3.update_xaxes(showgrid=False)
fig3.update_yaxes(showgrid=False, 
                 zeroline=True, zerolinecolor='black', zerolinewidth=3,
                 showticklabels=False)
fig3.update_layout(height=200, plot_bgcolor='white')
fig3.show()

fig4 = go.Figure()
x4 = three_LH["metallicity"]
fig4.add_trace(go.Scatter(
    x=x4, y=[0] * len(x4), mode='markers', marker_size=20,
))
fig4.update_xaxes(showgrid=False)
fig4.update_yaxes(showgrid=False, 
                 zeroline=True, zerolinecolor='black', zerolinewidth=3,
                 showticklabels=False)
fig4.update_layout(height=200, plot_bgcolor='white')
fig4.show()

fig2 = go.Figure()
x2 = three_sobol["metallicity"]
fig2.add_trace(go.Scatter(
    x=x2, y=[0] * len(x2), mode='markers', marker_size=20,
))
fig2.update_xaxes(showgrid=False)
fig2.update_yaxes(showgrid=False, 
                 zeroline=True, zerolinecolor='black', zerolinewidth=3,
                 showticklabels=False)
fig2.update_layout(height=200, plot_bgcolor='white')
fig2.show()

fig5 = go.Figure()
x5 = three_halton["metallicity"]
fig5.add_trace(go.Scatter(
    x=x5, y=[0] * len(x2), mode='markers', marker_size=20,
))
fig5.update_xaxes(showgrid=False)
fig5.update_yaxes(showgrid=False, 
                 zeroline=True, zerolinecolor='black', zerolinewidth=3,
                 showticklabels=False)
fig5.update_layout(height=200, plot_bgcolor='white')
fig5.show()


In [61]:
# https://community.plotly.com/t/plotting-a-simple-1d-number-line/39169/4
import plotly.graph_objects as go
fig = go.Figure()
x = three_LH["metallicity"]
fig.add_trace(go.Scatter(
    x=x, y=[0] * len(x), mode='markers', marker_size=20,
))
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False, 
                 zeroline=True, zerolinecolor='black', zerolinewidth=3,
                 showticklabels=False)
fig.update_layout(height=200, plot_bgcolor='white')
fig.show()

fig3 = go.Figure()
x3 = three_LH["common_alpha"]
fig3.add_trace(go.Scatter(
    x=x3, y=[0] * len(x3), mode='markers', marker_size=20,
))
fig3.update_xaxes(showgrid=False)
fig3.update_yaxes(showgrid=False, 
                 zeroline=True, zerolinecolor='black', zerolinewidth=3,
                 showticklabels=False)
fig3.update_layout(height=200, plot_bgcolor='white')
fig3.show()

fig4 = go.Figure()
x4 = three_LH["sigma_bh"]
fig4.add_trace(go.Scatter(
    x=x4, y=[0] * len(x4), mode='markers', marker_size=20,
))
fig4.update_xaxes(showgrid=False)
fig4.update_yaxes(showgrid=False, 
                 zeroline=True, zerolinecolor='black', zerolinewidth=3,
                 showticklabels=False)
fig4.update_layout(height=200, plot_bgcolor='white')
fig4.show()

fig2 = go.Figure()
x2 = three_LH["sigma_sn"]
fig2.add_trace(go.Scatter(
    x=x2, y=[0] * len(x2), mode='markers', marker_size=20,
))
fig2.update_xaxes(showgrid=False)
fig2.update_yaxes(showgrid=False, 
                 zeroline=True, zerolinecolor='black', zerolinewidth=3,
                 showticklabels=False)
fig2.update_layout(height=200, plot_bgcolor='white')
fig2.show()




In [62]:
# https://community.plotly.com/t/rotating-3d-plots-with-plotly/34776/2
# https://community.plotly.com/t/how-to-export-animation-and-save-it-in-a-video-format-like-mp4-mpeg-or/64621/2
import plotly.graph_objects as go
import numpy as np
import plotly.io as pio

x, y, z = three_halton["metallicity"], three_halton["common_alpha"], three_halton["sigma_bh"]

fig = go.Figure(go.Scatter3d(x=x, y=y, z=z, mode="markers"))

x_eye = -1.25
y_eye = 2
z_eye = 1.0

fig.update_layout(
    title="Animation Test",
    width=600,
    height=600,
    scene_camera_eye=dict(x=x_eye, y=y_eye, z=z_eye),
    updatemenus=[
        dict(
            type="buttons",
            showactive=False,
            y=1,
            x=0.8,
            xanchor="left",
            yanchor="bottom",
            pad=dict(t=45, r=10),
            buttons=[
                dict(
                    label="Play",
                    method="animate",
                    args=[
                        None,
                        dict(
                            frame=dict(duration=5, redraw=True),
                            transition=dict(duration=0),
                            fromcurrent=True,
                            mode="immediate",
                        ),
                    ],
                )
            ],
        )
    ],
)


def rotate_z(x, y, z, sigma_bh):
    w = x + 1j * y
    return np.real(np.exp(1j * sigma_bh) * w), np.imag(np.exp(1j * sigma_bh) * w), z


frames = []
pil_frames = []
for t in np.arange(0, 3.14, 0.025):
    xe, ye, ze = rotate_z(x_eye, y_eye, z_eye, -t)
    frames.append(go.Frame(layout=dict(scene_camera_eye=dict(x=xe, y=ye, z=ze))))
fig.frames = frames

fig.show()

In [63]:
dim_discrepancies = []
# sample_dfs = []
dim_nums = [2, 3, 4,5]
num_samples = 1000
for name, sampling_fn in sampling_fns.items():
    for num_dims in dim_nums:
        bounds = {f"x{i+1}": [0, 1] for i in range(num_dims)}
        sample_df = sampling_fn(bounds, num_samples)
        discrepancy = qmc.discrepancy(sample_df.values)
        dim_discrepancies.append(dict(name=name, num_samples=sample_df.shape[0], discrepancy=discrepancy, num_dims=num_dims))
        # sample_dfs.append(sample_df)

dim_discrepancy_df = pd.DataFrame(dim_discrepancies)
pd.pivot_table(
    dim_discrepancy_df.drop("num_samples", axis=1),
    index=["num_dims", "discrepancy", "name"],
)


Empty DataFrame
Columns: []
Index: [(2, 1.6274119434278589e-06, latin_hypercube), (2, 1.655082743745595e-06, sobol), (2, 6.469601514957901e-06, halton), (2, 0.00017733449320922468, random), (2, 0.0003923703424881797, grid), (3, 4.538395524056327e-06, sobol), (3, 8.905136304582228e-06, latin_hypercube), (3, 1.246092214901573e-05, halton), (3, 0.0004852356628999299, random), (3, 0.008351056258295353, grid), (4, 1.3498016167234894e-05, sobol), (4, 2.485417009112645e-05, latin_hypercube), (4, 3.077747067714576e-05, halton), (4, 0.0007819969412059535, random), (4, 0.044111428795331475, grid), (5, 3.3291895294551566e-05, sobol), (5, 6.398263419349348e-05, latin_hypercube), (5, 6.469162181343968e-05, halton), (5, 0.0013936105895215878, random), (5, 0.2013654873759494, grid)]

In [64]:
dim_discrepancies = []
# sample_dfs = []
dim_nums = [2, 3,4,5]
num_samples = 1000
for name, sampling_fn in sampling_fns.items():
    for num_dims in dim_nums:
        bounds = {f"x{i+1}": [0, 1] for i in range(num_dims)}
        sample_df = sampling_fn(bounds, num_samples)
        discrepancy = qmc.discrepancy(sample_df.values)
        dim_discrepancies.append(dict(name=name, num_samples=sample_df.shape[0], discrepancy=discrepancy, num_dims=num_dims))
        # sample_dfs.append(sample_df)

dim_discrepancy_df = pd.DataFrame(dim_discrepancies)
pd.pivot_table(
    dim_discrepancy_df.drop("num_samples", axis=1),
    index=["num_dims", "discrepancy", "name"],
)


/global/cfs/projectdirs/katrin/users/aschuetz/software/conda_envs/resum/lib/python3.10/site-packages/scipy/stats/_qmc.py:958: UserWarning:

The balance properties of Sobol' points require n to be a power of 2.



Empty DataFrame
Columns: []
Index: [(2, 1.5100492520847553e-06, latin_hypercube), (2, 1.6067345656178844e-06, sobol), (2, 6.469601514957901e-06, halton), (2, 0.0003923703424881797, grid), (2, 0.00045029608639213237, random), (3, 4.70711212496866e-06, sobol), (3, 8.392296699133439e-06, latin_hypercube), (3, 1.246092214901573e-05, halton), (3, 0.0007948714852685779, random), (3, 0.008351056258295353, grid), (4, 1.310182096214696e-05, sobol), (4, 2.489366448443775e-05, latin_hypercube), (4, 3.077747067714576e-05, halton), (4, 0.0008881880735889247, random), (4, 0.044111428795331475, grid), (5, 3.300883383716702e-05, sobol), (5, 6.384910433854252e-05, latin_hypercube), (5, 6.469162181343968e-05, halton), (5, 0.0012069822749518622, random), (5, 0.2013654873759494, grid)]

In [65]:
dim_discrepancies = []
# sample_dfs = []
dim_nums = [2, 3, 4,5]
num_samples =1000
for name, sampling_fn in sampling_fns.items():
    for num_dims in dim_nums:
        bounds = {f"x{i+1}": [0, 1] for i in range(num_dims)}
        sample_df = sampling_fn(bounds, num_samples)
        discrepancy = qmc.discrepancy(sample_df.values)
        dim_discrepancies.append(dict(name=name, num_samples=sample_df.shape[0], discrepancy=discrepancy, num_dims=num_dims))
        # sample_dfs.append(sample_df)

dim_discrepancy_df = pd.DataFrame(dim_discrepancies)
pd.pivot_table(
    dim_discrepancy_df.drop("num_samples", axis=1),
    index=["num_dims", "discrepancy", "name"],
)


/global/cfs/projectdirs/katrin/users/aschuetz/software/conda_envs/resum/lib/python3.10/site-packages/scipy/stats/_qmc.py:958: UserWarning:

The balance properties of Sobol' points require n to be a power of 2.



Empty DataFrame
Columns: []
Index: [(2, 1.6393898103483906e-06, latin_hypercube), (2, 1.7234714280167651e-06, sobol), (2, 6.469601514957901e-06, halton), (2, 0.0003923703424881797, grid), (2, 0.0005818741628274005, random), (3, 4.583000506874768e-06, sobol), (3, 1.0334640623055336e-05, latin_hypercube), (3, 1.246092214901573e-05, halton), (3, 0.0008044465449714711, random), (3, 0.008351056258295353, grid), (4, 1.3013517750293246e-05, sobol), (4, 2.8475150037232666e-05, latin_hypercube), (4, 3.077747067714576e-05, halton), (4, 0.0008577559788216504, random), (4, 0.044111428795331475, grid), (5, 3.1977164063112795e-05, sobol), (5, 6.469162181343968e-05, halton), (5, 6.550224185297715e-05, latin_hypercube), (5, 0.0012922931945649196, random), (5, 0.2013654873759494, grid)]